In [0]:

from pyspark.sql import functions as F

# Healthcare data engineering - Unit tests

CATALOG = "healthcare_demo"

BRONZE_APPOINTMENTS = f"{CATALOG}.bronze.appointments"
BRONZE_CAPACITY = f"{CATALOG}.bronze.capacity"
SILVER_REFERRALS = f"{CATALOG}.silver.patient_referral_details"
GOLD_WAITING_SUMMARY = f"{CATALOG}.gold.waiting_time_summary"
GOLD_RECOMMENDATIONS = f"{CATALOG}.gold.appointment_recommendations"

test_results = []

def run_test(test_name, test_function):
    try:
        test_function()
        test_results.append((test_name, "PASS", None))
        print(f"PASS: {test_name}")
    except Exception as error:
        test_results.append((test_name, "FAIL", str(error)))
        print(f"FAIL: {test_name} - {error}")


# --------------------------------------------------
# 1. BRONZE TESTS
# --------------------------------------------------

appointments = spark.table(BRONZE_APPOINTMENTS)
capacity = spark.table(BRONZE_CAPACITY)

def test_bronze_appointments_not_empty():
    assert appointments.count() > 0, "Appointments table is empty"

def test_bronze_appointment_ids_unique():
    duplicates = (
        appointments
        .groupBy("appointment_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    assert duplicates == 0, "Duplicate appointment IDs found"

def test_bronze_capacity_not_negative():
    invalid = capacity.filter(F.col("available_slots") < 0).count()
    assert invalid == 0, "Negative clinic capacity found"

def test_bronze_capacity_not_empty():
    assert capacity.count() > 0, "Capacity table is empty"


# --------------------------------------------------
# 2. SILVER TESTS
# --------------------------------------------------

silver = spark.table(SILVER_REFERRALS)

def test_silver_not_empty():
    assert silver.count() > 0, "Silver table is empty"

def test_silver_referral_ids_unique():
    duplicates = (
        silver
        .groupBy("referral_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    assert duplicates == 0, "Duplicate referral IDs found"

def test_silver_waiting_days_not_negative():
    invalid = silver.filter(F.col("waiting_days") < 0).count()
    assert invalid == 0, "Negative waiting days found"

def test_silver_priority_values():
    invalid = silver.filter(
        ~F.col("priority").isin("Urgent", "Routine")
        | F.col("priority").isNull()
    ).count()
    assert invalid == 0, "Invalid priority values found"

def test_silver_appointment_dates():
    invalid = silver.filter(
        F.col("appointment_date").isNotNull()
        & (F.col("appointment_date") < F.col("referral_date"))
    ).count()
    assert invalid == 0, "Appointment date before referral date"


# --------------------------------------------------
# 3. GOLD REPORTING TESTS
# --------------------------------------------------

waiting_summary = spark.table(GOLD_WAITING_SUMMARY)

def test_gold_waiting_summary_not_empty():
    assert waiting_summary.count() > 0, "Waiting summary is empty"

def test_gold_referral_totals_match_silver():
    silver_count = silver.select("referral_id").distinct().count()
    gold_count = waiting_summary.agg(
        F.sum("total_referrals")
    ).first()[0]

    assert silver_count == gold_count, (
        f"Silver has {silver_count} referrals, "
        f"Gold summary has {gold_count}"
    )

def test_gold_waiting_days_non_negative():
    invalid = waiting_summary.filter(
        (F.col("average_waiting_days") < 0)
        | (F.col("maximum_waiting_days") < 0)
    ).count()
    assert invalid == 0, "Invalid waiting time in Gold"


# --------------------------------------------------
# 4. PATIENT OPTIMISATION TESTS
# --------------------------------------------------

recommendations = spark.table(GOLD_RECOMMENDATIONS)

def test_recommendations_not_empty():
    assert recommendations.count() > 0, "Recommendations table is empty"

def test_recommendation_referrals_unique():
    duplicates = (
        recommendations
        .groupBy("referral_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    assert duplicates == 0, "Duplicate referral recommendations found"

def test_allocation_status_valid():
    invalid = recommendations.filter(
        ~F.col("allocation_status").isin(
            "Recommended",
            "Waiting for capacity"
        )
        | F.col("allocation_status").isNull()
    ).count()
    assert invalid == 0, "Invalid allocation status found"

def test_recommended_patients_have_location():
    invalid = recommendations.filter(
        (F.col("allocation_status") == "Recommended")
        & F.col("recommended_location_id").isNull()
    ).count()
    assert invalid == 0, "Recommended patients missing a location"

def test_recommended_patients_within_capacity():
    allocated = (
        recommendations
        .filter(F.col("allocation_status") == "Recommended")
        .groupBy("specialty")
        .agg(F.countDistinct("referral_id").alias("allocated_count"))
    )

    capacity_by_specialty = (
        recommendations
        .groupBy("specialty")
        .agg(F.max("available_slots").alias("available_slots"))
    )

    exceeded = (
        allocated
        .join(capacity_by_specialty, "specialty")
        .filter(F.col("allocated_count") > F.col("available_slots"))
        .count()
    )

    assert exceeded == 0, "Clinic capacity exceeded"


# --------------------------------------------------
# 5. RUN ALL TESTS
# --------------------------------------------------

run_test("Bronze appointments not empty", test_bronze_appointments_not_empty)
run_test("Bronze appointment IDs unique", test_bronze_appointment_ids_unique)
run_test("Bronze capacity not negative", test_bronze_capacity_not_negative)
run_test("Bronze capacity not empty", test_bronze_capacity_not_empty)

run_test("Silver table not empty", test_silver_not_empty)
run_test("Silver referral IDs unique", test_silver_referral_ids_unique)
run_test("Silver waiting days not negative", test_silver_waiting_days_not_negative)
run_test("Silver priority values valid", test_silver_priority_values)
run_test("Silver appointment dates valid", test_silver_appointment_dates)

run_test("Gold waiting summary not empty", test_gold_waiting_summary_not_empty)
run_test("Gold referral totals match Silver", test_gold_referral_totals_match_silver)
run_test("Gold waiting days non-negative", test_gold_waiting_days_non_negative)

run_test("Recommendations not empty", test_recommendations_not_empty)
run_test("Recommendation referrals unique", test_recommendation_referrals_unique)
run_test("Allocation statuses valid", test_allocation_status_valid)
run_test("Recommended patients have location", test_recommended_patients_have_location)
run_test("Recommended patients within capacity", test_recommended_patients_within_capacity)



# --------------------------------------------------
# 6. DISPLAY TEST RESULTS
# --------------------------------------------------

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

results_schema = StructType([
    StructField("test_name", StringType(), False),
    StructField("result", StringType(), False),
    StructField("error_message", StringType(), True)
])

results_df = spark.createDataFrame(
    test_results,
    schema=results_schema
)

display(results_df)

total_tests = len(test_results)
passed_tests = sum(
    1 for result in test_results if result[1] == "PASS"
)
failed_tests = total_tests - passed_tests

print(f"Total tests: {total_tests}")
print(f"Passed: {passed_tests}")
print(f"Failed: {failed_tests}")

if failed_tests > 0:
    raise Exception("Unit testing failed. Please review the failed tests.")

print("All healthcare pipeline unit tests passed.")


PASS: Bronze appointments not empty
PASS: Bronze appointment IDs unique
PASS: Bronze capacity not negative
PASS: Bronze capacity not empty
PASS: Silver table not empty
PASS: Silver referral IDs unique
PASS: Silver waiting days not negative
PASS: Silver priority values valid
PASS: Silver appointment dates valid
PASS: Gold waiting summary not empty
PASS: Gold referral totals match Silver
PASS: Gold waiting days non-negative
PASS: Recommendations not empty
PASS: Recommendation referrals unique
PASS: Allocation statuses valid
PASS: Recommended patients have location
PASS: Recommended patients within capacity


test_name,result,error_message
Bronze appointments not empty,PASS,null
Bronze appointment IDs unique,PASS,null
Bronze capacity not negative,PASS,null
Bronze capacity not empty,PASS,null
Silver table not empty,PASS,null
Silver referral IDs unique,PASS,null
Silver waiting days not negative,PASS,null
Silver priority values valid,PASS,null
Silver appointment dates valid,PASS,null
Gold waiting summary not empty,PASS,null


Total tests: 17
Passed: 17
Failed: 0
All healthcare pipeline unit tests passed.
